# Lesson 7: Perturb a model

**You will learn:** how to silence or scale units during a run, measure the effect on behavior, and undo the change exactly.
**Requires:** CPU only, about a minute.

Three objects describe a perturbation: `Selection` (which units), `Perturbation` (how: `'zero'`, `'scale'`, `'drive'`, `'replace'`) and `StateChange` (apply it, or undo it with `kind='reset'`). `kind='weights'` edits parameters and `kind='attention_heads'` edits attention heads in the same way.

**The question:** how much of the network can be silenced before the animal or vehicle readout fails? The readout is fitted on the intact network and then tested on the damaged one (`readout_fit='intact'`), as an animal trained before a lesion is tested after it.

In [1]:
import warnings
from functools import partialmethod
from tqdm.auto import tqdm
tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)   # no progress bars
warnings.filterwarnings('ignore')                            # keep the lesson output short

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# 120 natural images (CIFAR-10 test set, 12 per category), committed with the lessons.
ASSETS = Path('../assets/lessons')
meta = pd.read_csv(ASSETS / 'images.csv')
stimuli = StimulusSet([{'stimulus_id': row.file[:-4], 'image_path': str(ASSETS / 'images' / row.file),
                        'category': row.category, 'label': row.kind} for row in meta.itertuples()])
stimuli.identifier = 'cifar10-lessons-120'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli['label'].value_counts().to_dict()

{'animal': 72, 'vehicle': 48}

In [3]:
import torch
import torchvision
from PIL import Image

# SqueezeNet 1.1 trained on ImageNet (committed, 5 MB). It never saw these images.
net = torchvision.models.squeezenet1_1()
net.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth'))
net.eval()

TRANSFORM = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def preprocess(paths):
    """Image files to a (batch, 3, 224, 224) array, as the network was trained."""
    return np.stack([TRANSFORM(Image.open(p).convert('RGB')).numpy() for p in paths])

In [4]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper
from brainscore.perturbation import build_pytorch_ablation_fn

def squeezenet_subject(network, name):
    return BrainScoreModel(
        identifier=name, model=network,
        extractor={'vision': VisionWrapper(identifier=name, model=network, preprocessing=preprocess)},
        region_layer_map={'V1': 'features.2', 'V4': 'features.5', 'IT': 'features.12'},
        behavioral_readout_layer='features.12',
        intervention=build_pytorch_ablation_fn(network))

subject = squeezenet_subject(net, 'squeezenet1_1')
sorted(c for c in subject.out_channels if not c.startswith('behavior:'))

['behavior', 'neural:IT', 'neural:V1', 'neural:V4', 'perturbation']

In [5]:
# Half the images (6 per category) fit readouts; the other half test them.
order = stimuli.groupby('category').cumcount()
train = StimulusSet(stimuli[order < 6].reset_index(drop=True)); train.identifier = 'lessons-train'
test = StimulusSet(stimuli[order >= 6].reset_index(drop=True)); test.identifier = 'lessons-test'
for part in (train, test):
    part.stimulus_paths = dict(zip(part['stimulus_id'], part['image_path']))
len(train), len(test)

(60, 60)

In [6]:
from brainscore_core import Perturbation, Selection, StateChange, TaskContext
from brainscore_core.responses import Choice
from brainscore_core.streaming import BufferedSession
from brainscore_core.streaming_helpers import apply_state_change

task = TaskContext(response=Choice(['animal', 'vehicle']), fitting_stimuli=train,
                   prefer_path='readout', readout_fit='intact')

def accuracy():
    session = BufferedSession(stimulus_set=test, task_context=task, requested_output_channels=['behavior:choice'])
    subject.participate(session)
    chosen = np.array(['animal', 'vehicle'])[session.collect('behavior:choice').values.argmax(1)]
    return float(np.mean(chosen == np.asarray(test['label'])))

intact = accuracy()
print('intact:', round(intact, 3))

intact: 0.833


## Silence a growing share of one layer

`features.9` is a middle fire module with 384 output channels. Silence a random share of them, test, undo, and repeat with a larger share. Three random draws per share show the spread.

In [7]:
rng = np.random.default_rng(0)
rows = []
for share in (0.1, 0.25, 0.5, 0.75):
    for draw in range(3):
        channels = sorted(rng.choice(384, int(share * 384), replace=False).tolist())
        applied = apply_state_change(subject, StateChange(
            kind='ablation', target=Selection(layer='features.9', indices=channels, axis=1),
            perturbation=Perturbation(kind='zero')))
        rows.append({'share silenced': share, 'accuracy': accuracy()})
        apply_state_change(subject, StateChange(kind='reset', handle_id=applied.handle_id))
dose = pd.DataFrame(rows).groupby('share silenced')['accuracy'].agg(['mean', 'min', 'max'])
dose.round(3)

,mean,min,max
share silenced,,,
0.10,0.828,0.817,0.833
0.25,0.811,0.783,0.867
0.50,0.733,0.617,0.800
0.75,0.611,0.567,0.650


In [8]:
print('after undoing every change:', round(accuracy(), 3), '(intact was', round(intact, 3), ')')
subject.reset()

after undoing every change: 0.833 (intact was 0.833 )


How quickly accuracy falls, and whether it falls at all before half the layer is gone, is the network's robustness to this lesion. Undoing restores exactly the intact answers.

## Summary

- `Selection` says which units, `Perturbation` how, `StateChange` applies or undoes.
- Fit the readout intact, test the damaged network, and always undo (`reset()`).
- Random draws give the spread; [Lesson 8](08_choose_units.ipynb) chooses units by what they respond to.